# 16 · Fine-Tuning TimesFM with LoRA (Overview)

Zero-shot is enough for most cases, but when you have a lot of history in a
specific domain you can **fine-tune** for extra accuracy. The repo ships a
complete LoRA (PEFT) example using HuggingFace Transformers.

> Full runnable script:
> [`../timesfm-forecasting/examples/finetuning/`](../timesfm-forecasting/examples/finetuning/)

This notebook explains **when** and **how**, not a from-scratch training loop
(fine-tuning needs a GPU and the `transformers` + `peft` stack).

## When to fine-tune

| Situation | Recommendation |
| --------- | -------------- |
| General forecasting, mixed domains | **Zero-shot** (no fine-tuning) |
| One narrow domain, lots of history | Fine-tune with **LoRA** |
| Need a private model per client | Fine-tune per client → premium tier |
| Very little data | Zero-shot + covariates (notebook 10) |

LoRA trains only small adapter matrices — fast, cheap, and you keep the base
weights frozen, so you can ship one base model + many tiny adapters.

In [ ]:
# Dependencies for the fine-tuning example (GPU recommended)
%pip install -q transformers peft datasets accelerate

## Typical LoRA flow (pseudocode — see the example script for the real thing)

```python
from transformers import AutoModelForCausalLM
from peft import LoraConfig, get_peft_model

base = AutoModelForCausalLM.from_pretrained("google/timesfm-2.5-200m-pytorch")

lora_cfg = LoraConfig(
    r=8, lora_alpha=16, lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],   # attention projections
)
model = get_peft_model(base, lora_cfg)
model.print_trainable_parameters()   # only a tiny % is trainable

# ... standard Trainer loop on your (context -> horizon) windows ...
model.save_pretrained("timesfm-lora-mydomain")   # saves ONLY the adapter
```

### The business angle
- Ship the base model once; sell **per-client fine-tuned adapters** as a premium
  tier ("a model trained on *your* data").
- Adapters are a few MB → cheap to store and swap at inference time.

In [ ]:
# Explore the shipped example
import os
p = "../timesfm-forecasting/examples/finetuning"
print("files:", os.listdir(p))
print("\nRead the walkthrough:")
print(open(os.path.join(p, "README.md")).read()[:1200])